<a href="https://colab.research.google.com/github/shivani-swe/GC-UI-to-Code/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gradio as gr
import google.generativeai as genai
from google.colab import userdata
from PIL import Image
import tempfile

# 1. Authentication
try:
    API_KEY = userdata.get('Gemini_API_Key')
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel('gemini-3-flash-preview')
    print("✅ Gemini Model Initialized")
except Exception as e:
    print(f"❌ Error: Ensure 'GEMINI_API_KEY' is set in Colab Secrets. {e}")

# 2. Logic: Image to Tailwind
def process_screenshot(input_img, framework):
    if input_img is None:
        return "Please upload an image.", "", None

    prompt = f"You are a master developer. Recreate this UI to {framework} with Tailwind CSS. Output ONLY the code block."

    # Multimodal call: passing the prompt and the image
    response = model.generate_content([prompt, input_img])

    # Cleaning the response text
    code = response.text.replace("```html", "").replace("```jsx", "").replace("```", "").strip()

    ext= "index.html" if framework=="HTML" else "jsx"
    file_path= f"generated_ui.{ext}"
    with open(file_path, "w") as f:
      f.write(code)
    return code, code, file_path

# 3. Interface Design
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("AI UI-to-Code Prototype")

    with gr.Row():
        with gr.Column(scale=1):
          img_in= gr.Image(type="pil", label="1. Upload Screenschot")
          frame_choice= gr.Radio(['HTML','React', 'Vue'], value= "HTML", label= "2. Choose Framework")
          btn=gr.Button("Build UI", variant= "primary")
          file_out= gr.File(label="3. Download Output")


        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("Live Preview"):
                    preview = gr.HTML()
                with gr.TabItem("Export Code"):
                    code_out = gr.Code(language="html")

    btn.click(
        fn=process_screenshot,
        inputs=[img_in, frame_choice],
        outputs=[code_out, preview, file_out]
    )


demo.launch(debug=True, inline= False, share= True)

✅ Gemini Model Initialized


/tmp/ipykernel_21734/1725433695.py:36: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d9613ec4615be7252d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')